In [ ]:
# Programming for Data Science (Python & R)

## Análise de Vendas e Previsão de Atrasos — Olist

## 1. Objetivo

Este notebook desenvolve uma solução analítica para compreender padrões de vendas,
comportamento dos clientes e fatores relacionados a atrasos nas entregas do marketplace Olist.

O fluxo contempla preparação e qualidade dos dados, integração das nove tabelas,
engenharia de atributos, análise exploratória, visualizações e modelagem preditiva
de classificação, com comparação entre Regressão Logística e Random Forest.

A variável-alvo `delivered_late` identifica pedidos entregues após a data estimada.
Para evitar vazamento de dados, a modelagem utiliza apenas atributos disponíveis
antes da entrega.


## 2. Configuração e Validação dos Arquivos

Os caminhos do projeto são definidos de forma relativa à pasta do notebook.
Antes da leitura, é verificada a existência da pasta `Data` e dos nove arquivos CSV.


In [ ]:
from pathlib import Path

PASTA_ATUAL = Path.cwd()
PASTA_DADOS = (PASTA_ATUAL.parent / "Data").resolve()

print("Notebook:")
print(PASTA_ATUAL)

print("\nDados:")
print(PASTA_DADOS)

print("\nData existe:")
print(PASTA_DADOS.exists())

print("\nQuantidade de CSV:")
print(len(list(PASTA_DADOS.glob("*.csv"))))

In [ ]:
print("CSV dentro da pasta Trabalho:")
print(len(list(PASTA_ATUAL.glob("*.csv"))))

print("\nCSV dentro da pasta Data:")
print(len(list(PASTA_DADOS.glob("*.csv"))))

In [ ]:
arquivos_csv = sorted(PASTA_DADOS.glob("*.csv"))

if len(arquivos_csv) != 9:
    raise FileNotFoundError(
        f"Esperados 9 arquivos CSV na pasta Data, mas foram encontrados {len(arquivos_csv)}."
    )

for i, arquivo in enumerate(arquivos_csv, start=1):
    print(f"{i}. {arquivo.name}")

print("\nValidação concluída: os 9 CSVs foram encontrados.")


### 2.1 Carregamento das Nove Tabelas

As tabelas são carregadas separadamente, preservando suas granularidades originais.
Campos de data são convertidos durante a leitura para permitir análises temporais.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

customers = pd.read_csv(
    PASTA_DADOS / "olist_customers_dataset.csv"
)

geolocation = pd.read_csv(
    PASTA_DADOS / "olist_geolocation_dataset.csv"
)

order_items = pd.read_csv(
    PASTA_DADOS / "olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"]
)

payments = pd.read_csv(
    PASTA_DADOS / "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    PASTA_DADOS / "olist_order_reviews_dataset.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

orders = pd.read_csv(
    PASTA_DADOS / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

products = pd.read_csv(
    PASTA_DADOS / "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    PASTA_DADOS / "olist_sellers_dataset.csv"
)

category_translation = pd.read_csv(
    PASTA_DADOS / "product_category_name_translation.csv"
)

print("As 9 tabelas foram carregadas com sucesso.")

In [ ]:
print("Customers:", customers.shape)
print("Geolocation:", geolocation.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Category Translation:", category_translation.shape)

## 3. Auditoria e Qualidade dos Dados

Nesta etapa são analisadas as dimensões, tipos de dados, valores ausentes,
duplicidades e chaves utilizadas no relacionamento entre as tabelas.

O objetivo é identificar possíveis problemas antes da integração das bases.

In [ ]:
tabelas = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

print("Dicionário de tabelas criado com sucesso.")

In [ ]:
auditoria = []

for nome, df in tabelas.items():
    auditoria.append({
        "tabela": nome,
        "linhas": df.shape[0],
        "colunas": df.shape[1],
        "nulos": df.isna().sum().sum(),
        "duplicadas_exatas": df.duplicated().sum()
    })

auditoria = pd.DataFrame(auditoria)

auditoria

In [ ]:
for nome, df in tabelas.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    
    print(f"\n{'=' * 60}")
    print(f"TABELA: {nome.upper()}")
    print(f"{'=' * 60}")
    
    if len(nulos) == 0:
        print("Nenhum valor ausente.")
    else:
        print(nulos)

In [ ]:
resumo_nulos = []

for nome, df in tabelas.items():
    
    for coluna in df.columns:
        
        quantidade = df[coluna].isna().sum()
        
        if quantidade > 0:
            
            percentual = (
                quantidade / len(df) * 100
            )
            
            resumo_nulos.append({
                "tabela": nome,
                "coluna": coluna,
                "nulos": quantidade,
                "percentual": percentual
            })

resumo_nulos = pd.DataFrame(resumo_nulos)

resumo_nulos.sort_values(
    "percentual",
    ascending=False
)

In [ ]:
for nome, df in tabelas.items():
    
    print(f"\n{'=' * 60}")
    print(f"TABELA: {nome.upper()}")
    print(f"{'=' * 60}")
    
    print(df.dtypes)

In [ ]:
orders.dtypes

In [ ]:
duplicidades = []

for nome, df in tabelas.items():
    
    duplicidades.append({
        "tabela": nome,
        "linhas": len(df),
        "duplicadas_exatas": df.duplicated().sum()
    })

duplicidades = pd.DataFrame(duplicidades)

duplicidades

### Observação sobre duplicidades da geolocalização

A auditoria identifica um volume elevado de linhas idênticas em `geolocation`.
Como essa tabela não possui uma chave única por linha e repetições podem refletir
registros recorrentes de localização, elas não são removidas automaticamente nesta etapa.
Na integração geográfica, os registros são consolidados por prefixo de CEP utilizando
a mediana de latitude e longitude, produzindo um ponto representativo por CEP.


### 3.1 Validação das chaves

As chaves das tabelas cadastrais são verificadas para garantir
a integridade dos relacionamentos antes da realização dos merges.

In [ ]:
print("ORDERS")
print(
    "order_id duplicados:",
    orders["order_id"].duplicated().sum()
)

print("\nCUSTOMERS")
print(
    "customer_id duplicados:",
    customers["customer_id"].duplicated().sum()
)

print("\nPRODUCTS")
print(
    "product_id duplicados:",
    products["product_id"].duplicated().sum()
)

print("\nSELLERS")
print(
    "seller_id duplicados:",
    sellers["seller_id"].duplicated().sum()
)

print("\nCATEGORY TRANSLATION")
print(
    "product_category_name duplicados:",
    category_translation[
        "product_category_name"
    ].duplicated().sum()
)

In [ ]:
print("ORDER_ITEMS")
print(
    "order_id repetidos:",
    order_items["order_id"].duplicated().sum()
)

print("\nPAYMENTS")
print(
    "order_id repetidos:",
    payments["order_id"].duplicated().sum()
)

print("\nREVIEWS")
print(
    "order_id repetidos:",
    reviews["order_id"].duplicated().sum()
)

In [ ]:
print(
    "customer_id únicos:",
    customers["customer_id"].nunique()
)

print(
    "customer_unique_id únicos:",
    customers["customer_unique_id"].nunique()
)

## 4. Limpeza e Tratamento de Inconsistências

Valores fisicamente impossíveis são identificados antes da modelagem.
Outliers plausíveis são preservados, pois podem representar operações reais.

### Estratégia para valores extremos

Valores fisicamente impossíveis são convertidos para ausentes. Valores extremos
plausíveis são preservados, pois podem representar operações reais. Na modelagem,
variáveis assimétricas recebem transformação logarítmica e as variáveis numéricas
são escaladas com `RobustScaler`. Nos boxplots, limites percentuais são usados apenas
para melhorar a visualização, sem excluir observações da base analítica.


In [ ]:
auditoria_impossiveis = {
    "peso_negativo":
        (products["product_weight_g"] < 0).sum(),
        
    "comprimento_negativo":
        (products["product_length_cm"] < 0).sum(),
        
    "altura_negativa":
        (products["product_height_cm"] < 0).sum(),
        
    "largura_negativa":
        (products["product_width_cm"] < 0).sum(),
        
    "preco_negativo":
        (order_items["price"] < 0).sum(),
        
    "frete_negativo":
        (order_items["freight_value"] < 0).sum(),
        
    "pagamento_negativo":
        (payments["payment_value"] < 0).sum(),
        
    "parcelas_zero_ou_negativas":
        (payments["payment_installments"] <= 0).sum()
}

pd.Series(
    auditoria_impossiveis,
    name="quantidade"
)

In [ ]:
colunas_fisicas = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for coluna in colunas_fisicas:
    
    products.loc[
        products[coluna] < 0,
        coluna
    ] = np.nan

In [ ]:
order_items.loc[
    order_items["price"] < 0,
    "price"
] = np.nan

order_items.loc[
    order_items["freight_value"] < 0,
    "freight_value"
] = np.nan

In [ ]:
payments.loc[
    payments["payment_value"] < 0,
    "payment_value"
] = np.nan

In [ ]:
payments.loc[
    payments["payment_installments"] <= 0,
    "payment_installments"
] = np.nan

In [ ]:
products[
    "product_category_name"
] = (
    products[
        "product_category_name"
    ]
    .fillna("desconhecida")
)

In [ ]:
products[
    "product_category_name"
].isna().sum()

In [ ]:
PASTA_SAIDA = Path.cwd() / "outputs"

PASTA_FIGURAS = (
    PASTA_SAIDA / "figuras"
)

PASTA_SAIDA.mkdir(
    exist_ok=True
)

PASTA_FIGURAS.mkdir(
    exist_ok=True
)

print("Pasta de saída:")
print(PASTA_SAIDA)

## 5. Integração das Bases e Engenharia de Atributos

Nesta etapa, as tabelas são integradas respeitando suas diferentes
granularidades. A geolocalização é resumida por prefixo de CEP e
utilizada para estimar a distância entre clientes e vendedores.

Itens, pagamentos e avaliações são agregados por pedido antes dos merges,
evitando multiplicação indevida de registros.

In [ ]:
geo_cep = (
    geolocation
    .groupby(
        "geolocation_zip_code_prefix",
        as_index=False
    )
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median")
    )
)

geo_cep.head()

In [ ]:
print("Geolocation original:", geolocation.shape)
print("Geolocation resumida:", geo_cep.shape)

In [ ]:
itens = order_items.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Order items original:", order_items.shape)
print("Após produtos:", itens.shape)

In [ ]:
itens = itens.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

itens[
    [
        "product_category_name",
        "product_category_name_english"
    ]
].head()

In [ ]:
itens["product_category_name_english"] = (
    itens["product_category_name_english"]
    .fillna("desconhecida")
)

In [ ]:
itens = itens.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("Após sellers:", itens.shape)

In [ ]:
itens[
    [
        "order_id",
        "product_id",
        "seller_id",
        "seller_state",
        "price",
        "freight_value"
    ]
].head()

In [ ]:
itens["product_volume_cm3"] = (
    itens["product_length_cm"]
    * itens["product_height_cm"]
    * itens["product_width_cm"]
)

In [ ]:
itens[
    [
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3"
    ]
].head()

In [ ]:
pedido_cliente = orders[
    [
        "order_id",
        "customer_id"
    ]
].merge(
    customers[
        [
            "customer_id",
            "customer_zip_code_prefix",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

pedido_cliente.head()

In [ ]:
itens = itens.merge(
    pedido_cliente[
        [
            "order_id",
            "customer_zip_code_prefix",
            "customer_state"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
geo_cliente = geo_cep.rename(
    columns={
        "geolocation_zip_code_prefix":
            "customer_zip_code_prefix",
        "latitude":
            "customer_lat",
        "longitude":
            "customer_lng"
    }
)

In [ ]:
itens = itens.merge(
    geo_cliente,
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

In [ ]:
itens[
    [
        "customer_zip_code_prefix",
        "customer_lat",
        "customer_lng"
    ]
].head()

In [ ]:
geo_vendedor = geo_cep.rename(
    columns={
        "geolocation_zip_code_prefix":
            "seller_zip_code_prefix",
        "latitude":
            "seller_lat",
        "longitude":
            "seller_lng"
    }
)

In [ ]:
itens = itens.merge(
    geo_vendedor,
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

In [ ]:
itens[
    [
        "seller_zip_code_prefix",
        "seller_lat",
        "seller_lng"
    ]
].head()

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calcula a distância geodésica aproximada entre dois pontos,
    em quilômetros, utilizando a fórmula de Haversine.
    """
    
    R = 6371.0
    
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    
    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(delta_lon / 2) ** 2
    )
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [ ]:
itens["distance_km"] = haversine(
    itens["customer_lat"],
    itens["customer_lng"],
    itens["seller_lat"],
    itens["seller_lng"]
)

In [ ]:
itens[
    [
        "customer_state",
        "seller_state",
        "distance_km"
    ]
].head(10)

In [ ]:
itens["distance_km"].describe()

In [ ]:
print(
    "Distâncias ausentes:",
    itens["distance_km"].isna().sum()
)

print(
    "Percentual:",
    itens["distance_km"].isna().mean() * 100
)

In [ ]:
def moda_ou_nan(serie):
    moda = serie.dropna().mode()
    
    if len(moda) == 0:
        return np.nan
    
    return moda.iloc[0]

In [ ]:
# Agregações numéricas por pedido (operações vetorizadas)
itens_pedido = (
    itens
    .groupby("order_id", as_index=False)
    .agg(
        quantidade_itens=("order_item_id", "count"),
        quantidade_vendedores=("seller_id", "nunique"),
        quantidade_categorias=("product_category_name_english", "nunique"),
        preco_total=("price", "sum"),
        frete_total=("freight_value", "sum"),
        peso_medio=("product_weight_g", "mean"),
        volume_medio=("product_volume_cm3", "mean"),
        distancia_media_km=("distance_km", "mean"),
        distancia_maxima_km=("distance_km", "max")
    )
)

# Moda da categoria por pedido. Em empates, usa ordem alfabética,
# equivalente ao primeiro valor retornado por Series.mode().
categoria_principal = (
    itens[["order_id", "product_category_name_english"]]
    .dropna()
    .groupby(["order_id", "product_category_name_english"], as_index=False)
    .size()
    .sort_values(
        ["order_id", "size", "product_category_name_english"],
        ascending=[True, False, True]
    )
    .drop_duplicates("order_id")
    [["order_id", "product_category_name_english"]]
    .rename(columns={"product_category_name_english": "categoria_principal"})
)

# Moda do estado do vendedor por pedido, com a mesma regra de desempate.
estado_vendedor_principal = (
    itens[["order_id", "seller_state"]]
    .dropna()
    .groupby(["order_id", "seller_state"], as_index=False)
    .size()
    .sort_values(
        ["order_id", "size", "seller_state"],
        ascending=[True, False, True]
    )
    .drop_duplicates("order_id")
    [["order_id", "seller_state"]]
    .rename(columns={"seller_state": "estado_vendedor"})
)

itens_pedido = (
    itens_pedido
    .merge(categoria_principal, on="order_id", how="left", validate="one_to_one")
    .merge(estado_vendedor_principal, on="order_id", how="left", validate="one_to_one")
)


In [ ]:
itens_pedido.head()

In [ ]:
print("Itens originais:", len(order_items))
print("Pedidos após agregação:", len(itens_pedido))
print(
    "order_id duplicados:",
    itens_pedido["order_id"].duplicated().sum()
)

In [ ]:
# Agregações numéricas de pagamento por pedido
pagamentos_pedido = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        valor_pagamento=("payment_value", "sum"),
        quantidade_pagamentos=("payment_sequential", "count"),
        parcelas_maximas=("payment_installments", "max")
    )
)

# Forma de pagamento modal por pedido; em empates, usa ordem alfabética.
tipo_pagamento_principal = (
    payments[["order_id", "payment_type"]]
    .dropna()
    .groupby(["order_id", "payment_type"], as_index=False)
    .size()
    .sort_values(
        ["order_id", "size", "payment_type"],
        ascending=[True, False, True]
    )
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "tipo_pagamento"})
)

pagamentos_pedido = pagamentos_pedido.merge(
    tipo_pagamento_principal,
    on="order_id",
    how="left",
    validate="one_to_one"
)

pagamentos_pedido.head()


In [ ]:
print(
    "Duplicados:",
    pagamentos_pedido[
        "order_id"
    ].duplicated().sum()
)

In [ ]:
avaliacoes_pedido = (
    reviews
    .groupby(
        "order_id",
        as_index=False
    )
    .agg(
        review_score=(
            "review_score",
            "mean"
        )
    )
)

avaliacoes_pedido.head()

## 6. Construção da Base Analítica

Após a agregação das tabelas de itens, pagamentos e avaliações,
os dados são integrados na granularidade de pedido.

Cada linha da base analítica passa a representar um pedido.

In [ ]:
base = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
base = base.merge(
    itens_pedido,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [ ]:
base = base.merge(
    pagamentos_pedido,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [ ]:
base = base.merge(
    avaliacoes_pedido,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [ ]:
print("Orders original:")
print(orders.shape)

print("\nBase analítica:")
print(base.shape)

print(
    "\nPedidos únicos:",
    base["order_id"].nunique()
)

print(
    "order_id duplicados:",
    base["order_id"].duplicated().sum()
)

In [ ]:
base.head()

In [ ]:
base.columns.tolist()

In [ ]:
modelo_df = base[
    (base["order_status"] == "delivered")
    & base[
        "order_delivered_customer_date"
    ].notna()
    & base[
        "order_estimated_delivery_date"
    ].notna()
].copy()

In [ ]:
print(
    "Pedidos na base:",
    len(base)
)

print(
    "Pedidos disponíveis para modelagem:",
    len(modelo_df)
)

In [ ]:
modelo_df["delivered_late"] = (
    modelo_df[
        "order_delivered_customer_date"
    ]
    >
    modelo_df[
        "order_estimated_delivery_date"
    ]
).astype(int)

In [ ]:
modelo_df[
    "delivered_late"
].value_counts()

In [ ]:
modelo_df[
    "delivered_late"
].value_counts(
    normalize=True
) * 100

In [ ]:
modelo_df["mes_compra"] = (
    modelo_df[
        "order_purchase_timestamp"
    ].dt.month
)

modelo_df["dia_semana_compra"] = (
    modelo_df[
        "order_purchase_timestamp"
    ].dt.dayofweek
)

modelo_df["hora_compra"] = (
    modelo_df[
        "order_purchase_timestamp"
    ].dt.hour
)

In [ ]:
modelo_df["prazo_prometido_dias"] = (
    modelo_df[
        "order_estimated_delivery_date"
    ]
    -
    modelo_df[
        "order_purchase_timestamp"
    ]
).dt.total_seconds() / 86400

In [ ]:
modelo_df["tempo_entrega_dias"] = (
    modelo_df[
        "order_delivered_customer_date"
    ]
    -
    modelo_df[
        "order_purchase_timestamp"
    ]
).dt.total_seconds() / 86400

In [ ]:
modelo_df["dias_atraso"] = (
    modelo_df[
        "order_delivered_customer_date"
    ]
    -
    modelo_df[
        "order_estimated_delivery_date"
    ]
).dt.total_seconds() / 86400

In [ ]:
modelo_df["dias_atraso"] = (
    modelo_df[
        "dias_atraso"
    ].clip(lower=0)
)

In [ ]:
modelo_df["frete_preco"] = (
    modelo_df["frete_total"]
    /
    modelo_df["preco_total"]
)

In [ ]:
modelo_df["frete_preco"] = (
    modelo_df["frete_preco"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

In [ ]:
modelo_df["mesmo_estado"] = (
    modelo_df["customer_state"]
    ==
    modelo_df["estado_vendedor"]
).astype(int)

In [ ]:
modelo_df["log_preco"] = np.log1p(
    modelo_df["preco_total"]
)

modelo_df["log_frete"] = np.log1p(
    modelo_df["frete_total"]
)

modelo_df["log_peso"] = np.log1p(
    modelo_df["peso_medio"]
)

modelo_df["log_volume"] = np.log1p(
    modelo_df["volume_medio"]
)

modelo_df["log_distancia"] = np.log1p(
    modelo_df["distancia_media_km"]
)

In [ ]:
print("BASE ANALÍTICA")
print("Linhas:", base.shape[0])
print("Colunas:", base.shape[1])

print("\nBASE DE MODELAGEM")
print("Linhas:", modelo_df.shape[0])
print("Colunas:", modelo_df.shape[1])

print("\nDISTRIBUIÇÃO DO ALVO")
print(
    modelo_df[
        "delivered_late"
    ].value_counts()
)

print("\nPERCENTUAL")
print(
    modelo_df[
        "delivered_late"
    ].value_counts(
        normalize=True
    ) * 100
)

## 7. Análise Exploratória dos Dados

A análise exploratória tem como objetivo identificar padrões temporais,
regionais, comerciais e logísticos associados aos pedidos e aos atrasos
nas entregas.

Nesta etapa são analisados indicadores gerais, sazonalidade, comportamento
dos clientes, categorias de produtos, distância, peso, frete, pagamentos
e avaliações.

In [ ]:
total_pedidos = modelo_df["order_id"].nunique()

clientes_unicos = modelo_df[
    "customer_unique_id"
].nunique()

taxa_atraso = (
    modelo_df["delivered_late"].mean() * 100
)

receita_aproximada = (
    modelo_df["preco_total"].sum()
)

valor_total_pago = (
    modelo_df["valor_pagamento"].sum()
)

ticket_medio = (
    modelo_df["valor_pagamento"].mean()
)

tempo_medio_entrega = (
    modelo_df["tempo_entrega_dias"].mean()
)

dias_medios_atraso = (
    modelo_df.loc[
        modelo_df["delivered_late"] == 1,
        "dias_atraso"
    ].mean()
)

nota_media = (
    modelo_df["review_score"].mean()
)

In [ ]:
print("INDICADORES GERAIS")
print("=" * 50)

print(
    f"Pedidos entregues analisados: "
    f"{total_pedidos:,}"
)

print(
    f"Clientes únicos: "
    f"{clientes_unicos:,}"
)

print(
    f"Taxa geral de atraso: "
    f"{taxa_atraso:.2f}%"
)

print(
    f"Valor dos produtos: "
    f"R$ {receita_aproximada:,.2f}"
)

print(
    f"Valor total pago: "
    f"R$ {valor_total_pago:,.2f}"
)

print(
    f"Ticket médio por pedido: "
    f"R$ {ticket_medio:,.2f}"
)

print(
    f"Tempo médio de entrega: "
    f"{tempo_medio_entrega:.2f} dias"
)

print(
    f"Dias médios de atraso: "
    f"{dias_medios_atraso:.2f} dias"
)

print(
    f"Nota média das avaliações: "
    f"{nota_media:.2f}"
)

In [ ]:
kpis = pd.DataFrame({
    "Indicador": [
        "Pedidos entregues analisados",
        "Clientes únicos",
        "Taxa geral de atraso (%)",
        "Valor dos produtos",
        "Valor total pago",
        "Ticket médio",
        "Tempo médio de entrega (dias)",
        "Dias médios de atraso",
        "Nota média"
    ],
    
    "Valor": [
        total_pedidos,
        clientes_unicos,
        taxa_atraso,
        receita_aproximada,
        valor_total_pago,
        ticket_medio,
        tempo_medio_entrega,
        dias_medios_atraso,
        nota_media
    ]
})

kpis

In [ ]:
modelo_df["ano_mes"] = (
    modelo_df["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

In [ ]:
evolucao_mensal = (
    modelo_df
    .groupby(
        "ano_mes",
        as_index=False
    )
    .agg(
        pedidos=("order_id", "count"),
        valor_produtos=("preco_total", "sum"),
        taxa_atraso=("delivered_late", "mean")
    )
)

evolucao_mensal["taxa_atraso"] *= 100

evolucao_mensal.head()

In [ ]:
evolucao_mensal

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=evolucao_mensal,
    x="ano_mes",
    y="pedidos",
    marker="o"
)

plt.title(
    "Evolução Mensal da Quantidade de Pedidos"
)

plt.xlabel("Ano/Mês")
plt.ylabel("Quantidade de Pedidos")

plt.xticks(
    rotation=60,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    PASTA_FIGURAS / "01_pedidos_mensais.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=evolucao_mensal,
    x="ano_mes",
    y="taxa_atraso",
    marker="o"
)

plt.title(
    "Evolução Mensal da Taxa de Atraso"
)

plt.xlabel("Ano/Mês")
plt.ylabel("Taxa de Atraso (%)")

plt.xticks(
    rotation=60,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    PASTA_FIGURAS / "02_taxa_atraso_mensal.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
pedidos_cliente = (
    modelo_df
    .groupby("customer_unique_id")
    ["order_id"]
    .nunique()
)

pedidos_cliente.describe()

In [ ]:
clientes_recorrentes = (
    pedidos_cliente > 1
).sum()

percentual_recorrentes = (
    clientes_recorrentes
    / len(pedidos_cliente)
    * 100
)

print(
    f"Clientes únicos: "
    f"{len(pedidos_cliente):,}"
)

print(
    f"Clientes com mais de um pedido: "
    f"{clientes_recorrentes:,}"
)

print(
    f"Percentual de clientes recorrentes: "
    f"{percentual_recorrentes:.2f}%"
)

In [ ]:
atraso_estado = (
    modelo_df
    .groupby(
        "customer_state"
    )
    .agg(
        pedidos=("order_id", "count"),
        atrasos=("delivered_late", "sum"),
        taxa_atraso=("delivered_late", "mean")
    )
)

atraso_estado["taxa_atraso"] *= 100

In [ ]:
atraso_estado_100 = (
    atraso_estado[
        atraso_estado["pedidos"] >= 100
    ]
    .sort_values(
        "taxa_atraso",
        ascending=False
    )
)

atraso_estado_100

In [ ]:
dados_estado_plot = atraso_estado_100.reset_index()

plt.figure(figsize=(12, 6))
plt.bar(
    dados_estado_plot["customer_state"],
    dados_estado_plot["taxa_atraso"]
)
plt.title("Taxa de Atraso por Estado do Cliente")
plt.xlabel("Estado")
plt.ylabel("Taxa de Atraso (%)")
plt.tight_layout()
plt.savefig(
    PASTA_FIGURAS / "03_atraso_por_estado.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
atraso_categoria = (
    modelo_df
    .groupby(
        "categoria_principal"
    )
    .agg(
        pedidos=("order_id", "count"),
        atrasos=("delivered_late", "sum"),
        taxa_atraso=("delivered_late", "mean")
    )
)

atraso_categoria["taxa_atraso"] *= 100

In [ ]:
atraso_categoria_100 = (
    atraso_categoria[
        atraso_categoria["pedidos"] >= 100
    ]
    .sort_values(
        "taxa_atraso",
        ascending=False
    )
)

atraso_categoria_100.head(15)

In [ ]:
top10_categoria_atraso = (
    atraso_categoria_100
    .head(10)
    .reset_index()
)

plt.figure(figsize=(12, 6))
plt.barh(
    top10_categoria_atraso["categoria_principal"],
    top10_categoria_atraso["taxa_atraso"]
)
plt.title("Top 10 Categorias por Taxa de Atraso")
plt.xlabel("Taxa de Atraso (%)")
plt.ylabel("Categoria")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(
    PASTA_FIGURAS / "04_atraso_por_categoria.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
distancia_atraso = (
    modelo_df
    .groupby("delivered_late")
    ["distancia_media_km"]
    .agg(
        ["count", "mean", "median"]
    )
)

distancia_atraso

In [ ]:
dados_distancia = modelo_df[
    modelo_df["distancia_media_km"].notna()
].copy()

limite_distancia = dados_distancia["distancia_media_km"].quantile(0.95)
dados_distancia_plot = dados_distancia[
    dados_distancia["distancia_media_km"] <= limite_distancia
]

grupo_prazo = dados_distancia_plot.loc[
    dados_distancia_plot["delivered_late"] == 0, "distancia_media_km"
]
grupo_atraso = dados_distancia_plot.loc[
    dados_distancia_plot["delivered_late"] == 1, "distancia_media_km"
]

plt.figure(figsize=(8, 6))
plt.boxplot(
    [grupo_prazo, grupo_atraso],
    tick_labels=["No prazo", "Atrasado"],
    showfliers=False
)
plt.title("Distância Cliente-Vendedor e Atraso")
plt.xlabel("Situação da entrega")
plt.ylabel("Distância média (km)")
plt.tight_layout()
plt.savefig(
    PASTA_FIGURAS / "05_distancia_atraso.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
peso_atraso = (
    modelo_df
    .groupby("delivered_late")
    ["peso_medio"]
    .agg(
        ["count", "mean", "median"]
    )
)

peso_atraso

In [ ]:
dados_peso = modelo_df[
    modelo_df["peso_medio"].notna()
].copy()

limite_peso = dados_peso["peso_medio"].quantile(0.95)
dados_peso_plot = dados_peso[
    dados_peso["peso_medio"] <= limite_peso
]

grupo_prazo_peso = dados_peso_plot.loc[
    dados_peso_plot["delivered_late"] == 0, "peso_medio"
]
grupo_atraso_peso = dados_peso_plot.loc[
    dados_peso_plot["delivered_late"] == 1, "peso_medio"
]

plt.figure(figsize=(8, 6))
plt.boxplot(
    [grupo_prazo_peso, grupo_atraso_peso],
    tick_labels=["No prazo", "Atrasado"],
    showfliers=False
)
plt.title("Peso Médio dos Produtos e Atraso")
plt.xlabel("Situação da entrega")
plt.ylabel("Peso médio (g)")
plt.tight_layout()
plt.savefig(
    PASTA_FIGURAS / "06_peso_atraso.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
avaliacao_atraso = (
    modelo_df
    .groupby(
        "delivered_late"
    )
    .agg(
        pedidos=("order_id", "count"),
        nota_media=("review_score", "mean")
    )
)

avaliacao_atraso

In [ ]:
avaliacao_atraso_plot = avaliacao_atraso.reset_index()

plt.figure(figsize=(7, 5))
plt.bar(
    avaliacao_atraso_plot["delivered_late"].astype(str),
    avaliacao_atraso_plot["nota_media"]
)
plt.title("Nota Média: Pedidos no Prazo versus Atrasados")
plt.xlabel("Entrega atrasada (0 = Não, 1 = Sim)")
plt.ylabel("Nota média")
plt.ylim(0, 5)
plt.tight_layout()
plt.savefig(
    PASTA_FIGURAS / "07_avaliacao_atraso.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
print("RESUMO DA EDA")
print("=" * 60)

print(
    f"Pedidos analisados: "
    f"{total_pedidos:,}"
)

print(
    f"Taxa geral de atraso: "
    f"{taxa_atraso:.2f}%"
)

print(
    f"Estado com maior taxa de atraso: "
    f"{atraso_estado_100.index[0]} "
    f"({atraso_estado_100.iloc[0]['taxa_atraso']:.2f}%)"
)

print(
    f"Categoria com maior taxa de atraso: "
    f"{atraso_categoria_100.index[0]} "
    f"({atraso_categoria_100.iloc[0]['taxa_atraso']:.2f}%)"
)

print(
    f"Tempo médio de entrega: "
    f"{tempo_medio_entrega:.2f} dias"
)

print(
    f"Nota média geral: "
    f"{nota_media:.2f}"
)

## 8. Modelagem Preditiva de Atrasos

O problema é tratado como uma classificação binária.

A variável-alvo `delivered_late` assume:

- 0: pedido entregue no prazo;
- 1: pedido entregue após a data estimada.

Para evitar vazamento de dados, somente informações disponíveis
até o momento da compra são utilizadas como variáveis preditoras.

São comparados dois algoritmos:

1. Regressão Logística;
2. Random Forest.

A avaliação utiliza divisão temporal dos dados, simulando a previsão
de pedidos futuros.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    TimeSeriesSplit,
    cross_validate
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

from sklearn.base import clone

import joblib

RANDOM_STATE = 42

print("Bibliotecas de Machine Learning carregadas.")

In [ ]:
modelo_df["log_valor_pagamento"] = np.log1p(
    modelo_df["valor_pagamento"]
)

In [ ]:
modelo_df = modelo_df.replace(
    [np.inf, -np.inf],
    np.nan
)

### 8.1 Seleção dos atributos

As variáveis utilizadas na previsão representam informações temporais,
comerciais, logísticas, geográficas e de pagamento disponíveis antes
da entrega do pedido.

Variáveis posteriores à entrega são excluídas da modelagem.

In [ ]:
variaveis_numericas = [
    "mes_compra",
    "dia_semana_compra",
    "hora_compra",
    "prazo_prometido_dias",
    "quantidade_itens",
    "quantidade_vendedores",
    "quantidade_categorias",
    "log_preco",
    "log_frete",
    "frete_preco",
    "log_peso",
    "log_volume",
    "log_distancia",
    "log_valor_pagamento",
    "quantidade_pagamentos",
    "parcelas_maximas",
    "mesmo_estado"
]

In [ ]:
variaveis_categoricas = [
    "customer_state",
    "estado_vendedor",
    "categoria_principal",
    "tipo_pagamento"
]

In [ ]:
features = (
    variaveis_numericas
    + variaveis_categoricas
)

print("Quantidade de variáveis preditoras:", len(features))

In [ ]:
variaveis_faltantes = [
    coluna
    for coluna in features
    if coluna not in modelo_df.columns
]

if variaveis_faltantes:
    print("Variáveis faltantes:")
    print(variaveis_faltantes)
else:
    print("Todas as variáveis foram encontradas.")

In [ ]:
variaveis_proibidas = [
    "review_score",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "tempo_entrega_dias",
    "dias_atraso"
]

vazamento = [
    coluna
    for coluna in variaveis_proibidas
    if coluna in features
]

print("Variáveis com possível vazamento:", vazamento)

In [ ]:
modelo_ordenado = (
    modelo_df
    .sort_values(
        "order_purchase_timestamp"
    )
    .reset_index(drop=True)
)

print(
    modelo_ordenado[
        "order_purchase_timestamp"
    ].min()
)

print(
    modelo_ordenado[
        "order_purchase_timestamp"
    ].max()
)

In [ ]:
X = modelo_ordenado[
    features
].copy()

In [ ]:
y = modelo_ordenado[
    "delivered_late"
].copy()

In [ ]:
print("X:", X.shape)
print("y:", y.shape)

print("\nDistribuição do alvo:")
print(y.value_counts())

print("\nPercentual:")
print(
    y.value_counts(
        normalize=True
    ) * 100
)

In [ ]:
corte = int(
    len(modelo_ordenado) * 0.80
)

X_train = X.iloc[:corte].copy()
X_test = X.iloc[corte:].copy()

y_train = y.iloc[:corte].copy()
y_test = y.iloc[corte:].copy()

In [ ]:
print("TREINO")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nTESTE")
print("X:", X_test.shape)
print("y:", y_test.shape)

In [ ]:
print("PERÍODO DE TREINO")
print(
    modelo_ordenado.loc[
        :corte - 1,
        "order_purchase_timestamp"
    ].min()
)

print(
    modelo_ordenado.loc[
        :corte - 1,
        "order_purchase_timestamp"
    ].max()
)

print("\nPERÍODO DE TESTE")
print(
    modelo_ordenado.loc[
        corte:,
        "order_purchase_timestamp"
    ].min()
)

print(
    modelo_ordenado.loc[
        corte:,
        "order_purchase_timestamp"
    ].max()
)

In [ ]:
print(
    f"Taxa de atraso no treino: "
    f"{y_train.mean() * 100:.2f}%"
)

print(
    f"Taxa de atraso no teste: "
    f"{y_test.mean() * 100:.2f}%"
)

### 8.2 Pipeline de pré-processamento

Valores numéricos ausentes são imputados pela mediana calculada
exclusivamente no conjunto de treino.

Em seguida é utilizado RobustScaler, que possui menor sensibilidade
a valores extremos.

In [ ]:
pipeline_numerico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            RobustScaler()
        )
    ]
)

In [ ]:
pipeline_categorico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessador = ColumnTransformer(
    transformers=[
        (
            "num",
            pipeline_numerico,
            variaveis_numericas
        ),
        (
            "cat",
            pipeline_categorico,
            variaveis_categoricas
        )
    ]
)

print("Pré-processador criado.")

### 8.3 Regressão Logística

A Regressão Logística é utilizada como modelo de referência
por sua interpretabilidade e capacidade de estimar probabilidades
para uma classificação binária.

In [ ]:
regressao_logistica = Pipeline(
    steps=[
        (
            "preprocessamento",
            preprocessador
        ),
        (
            "modelo",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ]
)

### 8.4 Random Forest

A Random Forest combina várias árvores de decisão e permite capturar
relações não lineares e interações entre os atributos.

In [ ]:
random_forest = Pipeline(
    steps=[
        (
            "preprocessamento",
            preprocessador
        ),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=150,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
tscv = TimeSeriesSplit(
    n_splits=5
)

print(tscv)

In [ ]:
modelos = {
    "Regressão Logística":
        regressao_logistica,
        
    "Random Forest":
        random_forest
}

In [ ]:
resultados_cv = []

for nome, modelo in modelos.items():
    
    print(
        f"Avaliando: {nome}"
    )
    
    scores = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=tscv,
        scoring={
            "roc_auc":
                "roc_auc",
            "pr_auc":
                "average_precision",
            "recall":
                "recall",
            "precision":
                "precision",
            "f1":
                "f1"
        },
        n_jobs=1
    )
    
    resultados_cv.append({
        "modelo": nome,
        
        "ROC_AUC":
            scores[
                "test_roc_auc"
            ].mean(),
        
        "PR_AUC":
            scores[
                "test_pr_auc"
            ].mean(),
        
        "Recall":
            scores[
                "test_recall"
            ].mean(),
        
        "Precision":
            scores[
                "test_precision"
            ].mean(),
        
        "F1":
            scores[
                "test_f1"
            ].mean()
    })

print("Validação concluída.")

In [ ]:
resultados_cv = pd.DataFrame(
    resultados_cv
)

resultados_cv

In [ ]:
resultados_cv = (
    resultados_cv
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

resultados_cv

In [ ]:
melhor_nome = (
    resultados_cv
    .iloc[0]["modelo"]
)

melhor_modelo = (
    modelos[melhor_nome]
)

print(
    "Melhor modelo pela PR-AUC:"
)

print(melhor_nome)

### 8.5 Avaliação Final e Escolha do Limiar

Após a seleção do algoritmo pela PR-AUC na validação temporal, o limiar de decisão
é escolhido exclusivamente nos dados de treino/validação pela métrica F2, que atribui
maior peso ao Recall. O conjunto de teste permanece isolado até a avaliação final.


In [ ]:
y_validacao = []
prob_validacao = []

for indice_treino, indice_validacao in tscv.split(X_train):
    
    modelo_fold = clone(melhor_modelo)
    
    modelo_fold.fit(
        X_train.iloc[indice_treino],
        y_train.iloc[indice_treino]
    )
    
    probabilidades = modelo_fold.predict_proba(
        X_train.iloc[indice_validacao]
    )[:, 1]
    
    prob_validacao.extend(probabilidades)
    
    y_validacao.extend(
        y_train.iloc[indice_validacao].values
    )

print("Previsões de validação concluídas.")

In [ ]:
y_validacao = np.array(y_validacao)
prob_validacao = np.array(prob_validacao)

print("Observações de validação:", len(y_validacao))
print("Probabilidades geradas:", len(prob_validacao))

In [ ]:
precisao_curve, recall_curve, thresholds = (
    precision_recall_curve(
        y_validacao,
        prob_validacao
    )
)

In [ ]:
beta = 2

f2_scores = (
    (1 + beta ** 2)
    * precisao_curve[:-1]
    * recall_curve[:-1]
    /
    (
        beta ** 2 * precisao_curve[:-1]
        + recall_curve[:-1]
        + 1e-10
    )
)

In [ ]:
indice_melhor = np.argmax(f2_scores)

melhor_threshold = thresholds[indice_melhor]
melhor_f2_validacao = f2_scores[indice_melhor]

print(f"Threshold escolhido: {melhor_threshold:.3f}")
print(f"Melhor F2 na validação: {melhor_f2_validacao:.3f}")

In [ ]:
melhor_modelo.fit(
    X_train,
    y_train
)

print(
    f"Modelo final treinado: {melhor_nome}"
)

In [ ]:
prob_teste = melhor_modelo.predict_proba(
    X_test
)[:, 1]

In [ ]:
pred_teste = (
    prob_teste >= melhor_threshold
).astype(int)

In [ ]:
print("Probabilidades:", len(prob_teste))
print("Previsões:", len(pred_teste))
print("Observações reais:", len(y_test))

In [ ]:
precision_final = precision_score(
    y_test,
    pred_teste
)

recall_final = recall_score(
    y_test,
    pred_teste
)

f1_final = f1_score(
    y_test,
    pred_teste
)

f2_final = fbeta_score(
    y_test,
    pred_teste,
    beta=2
)

roc_auc_final = roc_auc_score(
    y_test,
    prob_teste
)

pr_auc_final = average_precision_score(
    y_test,
    prob_teste
)

In [ ]:
print("RESULTADOS FINAIS NO TESTE")
print("=" * 50)

print(f"Modelo: {melhor_nome}")
print(f"Threshold: {melhor_threshold:.3f}")
print(f"Precision: {precision_final:.3f}")
print(f"Recall: {recall_final:.3f}")
print(f"F1: {f1_final:.3f}")
print(f"F2: {f2_final:.3f}")
print(f"ROC-AUC: {roc_auc_final:.3f}")
print(f"PR-AUC: {pr_auc_final:.3f}")

In [ ]:
baseline_pr_auc = y_test.mean()

print(
    f"Baseline PR-AUC: "
    f"{baseline_pr_auc:.3f}"
)

print(
    f"PR-AUC do modelo: "
    f"{pr_auc_final:.3f}"
)

In [ ]:
print(
    classification_report(
        y_test,
        pred_teste,
        digits=3
    )
)

In [ ]:
matriz = confusion_matrix(
    y_test,
    pred_teste
)

matriz

In [ ]:
plt.figure(figsize=(6, 5))

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False
)

plt.title("Matriz de Confusão")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")

plt.tight_layout()

plt.savefig(
    PASTA_FIGURAS / "08_matriz_confusao.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(
    y_test,
    prob_teste
)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"ROC-AUC = {roc_auc_final:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos")
plt.title("Curva ROC")

plt.legend()
plt.tight_layout()

plt.savefig(
    PASTA_FIGURAS / "09_curva_roc.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
precision_pr, recall_pr, _ = (
    precision_recall_curve(
        y_test,
        prob_teste
    )
)

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    recall_pr,
    precision_pr,
    label=f"PR-AUC = {pr_auc_final:.3f}"
)

plt.axhline(
    y=baseline_pr_auc,
    linestyle="--",
    label=f"Baseline = {baseline_pr_auc:.3f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall")

plt.legend()
plt.tight_layout()

plt.savefig(
    PASTA_FIGURAS
    / "10_curva_precision_recall.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Interpretação da Avaliação

A classe de atrasos é minoritária e apresenta mudança temporal relevante:
a taxa de atraso é de aproximadamente **8,82% no treino** e **5,29% no teste**.

A Regressão Logística foi selecionada na validação temporal. No teste, apresentou
**PR-AUC de 0,104**, superior ao baseline de aproximadamente **0,053**, e
**ROC-AUC de 0,685**. Com limiar de aproximadamente **0,335**, o modelo alcançou
**Recall de 0,955**, identificando a maior parte dos pedidos que efetivamente atrasaram.

Em contrapartida, a **Precision de 0,068** indica grande quantidade de falsos positivos.
Esse comportamento decorre da estratégia de priorizar Recall/F2: reduzir falsos negativos
exige aceitar mais alertas preventivos. A matriz de confusão confirma esse trade-off.

A PR-AUC média da validação temporal foi superior à observada no teste, o que, junto
à mudança na prevalência de atrasos, sugere possível mudança de distribuição ao longo
do tempo. Em aplicação operacional, o desempenho deve ser monitorado e o limiar deve
ser calibrado conforme o custo de falsos positivos e falsos negativos.


In [ ]:
preprocessador_final = (
    melhor_modelo
    .named_steps["preprocessamento"]
)

nomes_features = (
    preprocessador_final
    .get_feature_names_out()
)

print(
    "Quantidade de atributos "
    "após transformação:",
    len(nomes_features)
)

In [ ]:
if melhor_nome == "Random Forest":
    
    importancias = (
        melhor_modelo
        .named_steps["modelo"]
        .feature_importances_
    )
    
    importancia_df = pd.DataFrame({
        "variavel": nomes_features,
        "importancia": importancias
    })
    
    importancia_df = (
        importancia_df
        .sort_values(
            "importancia",
            ascending=False
        )
        .reset_index(drop=True)
    )
    
    display(
        importancia_df.head(20)
    )

In [ ]:
if melhor_nome == "Random Forest":
    
    top_importancias = (
        importancia_df
        .head(15)
        .sort_values("importancia")
    )
    
    plt.figure(figsize=(11, 7))
    
    plt.barh(
        top_importancias["variavel"],
        top_importancias["importancia"]
    )
    
    plt.title(
        "Principais Variáveis Associadas à Previsão"
    )
    
    plt.xlabel("Importância")
    
    plt.tight_layout()
    
    plt.savefig(
        PASTA_FIGURAS
        / "11_importancia_variaveis.png",
        dpi=300,
        bbox_inches="tight"
    )
    
    plt.show()

In [ ]:
if melhor_nome == "Regressão Logística":
    
    coeficientes = (
        melhor_modelo
        .named_steps["modelo"]
        .coef_[0]
    )
    
    importancia_df = pd.DataFrame({
        "variavel": nomes_features,
        "coeficiente": coeficientes,
        "impacto_absoluto":
            np.abs(coeficientes)
    })
    
    importancia_df = (
        importancia_df
        .sort_values(
            "impacto_absoluto",
            ascending=False
        )
        .reset_index(drop=True)
    )
    
    display(
        importancia_df.head(20)
    )

In [ ]:
metricas_finais = pd.DataFrame({
    "Métrica": [
        "Precision",
        "Recall",
        "F1",
        "F2",
        "ROC-AUC",
        "PR-AUC",
        "Baseline PR-AUC",
        "Threshold"
    ],
    
    "Valor": [
        precision_final,
        recall_final,
        f1_final,
        f2_final,
        roc_auc_final,
        pr_auc_final,
        baseline_pr_auc,
        melhor_threshold
    ]
})

metricas_finais

In [ ]:
estado_maior_atraso = (
    atraso_estado_100.index[0]
)

taxa_estado_maior = (
    atraso_estado_100.iloc[0][
        "taxa_atraso"
    ]
)

categoria_maior_atraso = (
    atraso_categoria_100.index[0]
)

taxa_categoria_maior = (
    atraso_categoria_100.iloc[0][
        "taxa_atraso"
    ]
)

In [ ]:
resultado_relatorio = pd.DataFrame({
    "Indicador": [
        "Pedidos entregues analisados",
        "Taxa geral de atraso",
        "Valor dos produtos",
        "Estado com maior taxa de atraso",
        "Categoria com maior taxa de atraso",
        "Melhor modelo",
        "PR-AUC no teste",
        "Recall no teste"
    ],
    
    "Resultado": [
        f"{total_pedidos:,}",
        f"{taxa_atraso:.2f}%",
        f"R$ {receita_aproximada:,.2f}",
        
        (
            f"{estado_maior_atraso} "
            f"({taxa_estado_maior:.2f}%)"
        ),
        
        (
            f"{categoria_maior_atraso} "
            f"({taxa_categoria_maior:.2f}%)"
        ),
        
        melhor_nome,
        f"{pr_auc_final:.3f}",
        f"{recall_final:.3f}"
    ]
})

resultado_relatorio

In [ ]:
resultados_cv.to_csv(
    PASTA_SAIDA / "comparacao_modelos.csv",
    index=False
)

In [ ]:
metricas_finais.to_csv(
    PASTA_SAIDA
    / "metricas_modelo_final.csv",
    index=False
)

In [ ]:
resultado_relatorio.to_csv(
    PASTA_SAIDA
    / "indicadores_relatorio.csv",
    index=False
)

In [ ]:
previsoes = (
    modelo_ordenado
    .iloc[corte:][
        [
            "order_id",
            "order_purchase_timestamp"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

previsoes[
    "classe_real"
] = y_test.reset_index(
    drop=True
)

previsoes[
    "probabilidade_atraso"
] = prob_teste

previsoes[
    "classe_prevista"
] = pred_teste

previsoes.head()

In [ ]:
previsoes.to_csv(
    PASTA_SAIDA
    / "previsoes_teste.csv",
    index=False
)

In [ ]:
importancia_df.to_csv(
    PASTA_SAIDA
    / "importancia_variaveis.csv",
    index=False
)

In [ ]:
joblib.dump(
    melhor_modelo,
    PASTA_SAIDA
    / "modelo_atraso_olist.joblib"
)

print("Modelo salvo com sucesso.")

## 9. Recomendações de Negócio

Com base na análise exploratória e na modelagem preditiva, recomenda-se:

- utilizar a probabilidade prevista de atraso para criar uma fila de
  priorização operacional;

- acompanhar preventivamente pedidos classificados como de maior risco;

- revisar prazos prometidos nas regiões e categorias que apresentaram
  maior incidência de atrasos;

- observar distância, peso, volume, frete e complexidade do pedido quando
  essas características aparecerem entre as variáveis relevantes;

- monitorar continuamente a taxa real de atraso, Recall e PR-AUC;

- revisar o limiar de classificação de acordo com o custo operacional
  de falsos positivos e falsos negativos;

- retreinar o modelo diante de deterioração de desempenho ou mudanças
  importantes no processo logístico.

In [ ]:
from IPython.display import display, Markdown

In [ ]:
texto_conclusao = f"""
## 10. Conclusão da Parte Prática

Foram analisados **{total_pedidos:,} pedidos entregues**,
com taxa geral de atraso de **{taxa_atraso:.2f}%**.

Entre os estados com pelo menos 100 pedidos,
**{estado_maior_atraso}** apresentou a maior taxa de atraso,
com **{taxa_estado_maior:.2f}%**.

A categoria **{categoria_maior_atraso}** apresentou a maior
taxa observada entre categorias com pelo menos 100 pedidos,
com **{taxa_categoria_maior:.2f}%**.

O algoritmo selecionado foi **{melhor_nome}**.

No conjunto de teste foram obtidos:

- **Precision:** {precision_final:.3f}
- **Recall:** {recall_final:.3f}
- **F1:** {f1_final:.3f}
- **F2:** {f2_final:.3f}
- **ROC-AUC:** {roc_auc_final:.3f}
- **PR-AUC:** {pr_auc_final:.3f}

O modelo pode apoiar a priorização operacional de pedidos
com maior risco de atraso. As associações observadas não devem
ser interpretadas automaticamente como relações causais.
"""

display(
    Markdown(texto_conclusao)
)

In [ ]:
print("ARQUIVOS GERADOS")
print("=" * 60)

for arquivo in sorted(
    PASTA_SAIDA.rglob("*")
):
    
    if arquivo.is_file():
        
        print(
            arquivo.relative_to(
                PASTA_SAIDA
            )
        )